<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/01_hello_openimpala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 1 of 7: Getting Started with OpenImpala

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

OpenImpala computes effective transport properties (diffusivity, tortuosity, conductivity) directly on 3D voxel images of porous microstructures. It uses finite differences on the voxel grid — no mesh generation required — and is parallelised via MPI through AMReX and HYPRE.

This tutorial introduces the core workflow using a synthetic microstructure generated with [PoreSpy](https://porespy.org/), a widely used library for porous media analysis. No external data files are needed.

**What you will learn:**
1. Generate a random 3D porous microstructure with PoreSpy.
2. Calculate volume fraction, check percolation, and compute tortuosity.
3. Run a parametric sweep of porosity vs. tortuosity.

**Prerequisites:** None — this is the starting point for the series.

In [ ]:
# Install OpenImpala, PoreSpy for structure generation, and Matplotlib for plots.
!pip install -q openimpala porespy matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded and ready.")

# OpenImpala relies on AMReX and MPI under the hood.  In a notebook we
# need the runtime to stay alive across cells, so we open the session
# manually.  In a script, use the context manager instead:
#     with oi.Session():
#         ...
session = oi.Session()
session.__enter__()

# Try importing PoreSpy for realistic structure generation.
# If unavailable, we fall back to plain NumPy (see below).
try:
    import porespy as ps
    HAS_PORESPY = True
    print("PoreSpy available — will use blobs() for structure generation.")
except ImportError:
    HAS_PORESPY = False
    print("PoreSpy not found — using NumPy random generation instead.")

## 1. Generating a Synthetic Microstructure

If PoreSpy is available, its `blobs` generator creates random two-phase structures that resemble real porous media (e.g. electrode microstructures from X-ray tomography). Otherwise we fall back to `np.random.choice`, which produces a simpler random structure but requires no extra dependencies.

We generate a $100^3$ voxel grid (~1 million voxels) with a target porosity of 0.40:
- **Phase 0** — solid matrix
- **Phase 1** — void / pore space

**Phase convention:** OpenImpala labels each voxel with an integer phase ID. The `phase` parameter in functions like `oi.tortuosity(data, phase=1)` tells the solver which phase to treat as the transport medium. The default is `phase=0`, so **always pass `phase=` explicitly** to avoid silently computing the tortuosity of the wrong phase. There is no "correct" phase numbering — it depends on how your image was segmented.

In [ ]:
N = 100
np.random.seed(42)  # For reproducibility

if HAS_PORESPY:
    # PoreSpy blobs: spatially correlated random structure
    micro = ps.generators.blobs(
        shape=[N, N, N],
        porosity=0.40,
        blobiness=1.5,
    ).astype(np.int32)
else:
    # Pure NumPy fallback: uncorrelated random voxels
    micro = np.random.choice(
        [0, 1], size=(N, N, N), p=[0.60, 0.40]
    ).astype(np.int32)

print(f"Microstructure shape: {micro.shape} ({micro.size / 1e6:.1f} million voxels)")
print(f"Measured porosity:    {micro.mean():.4f}")
print(f"Unique phase IDs:     {np.unique(micro)}  (0=solid, 1=pore)")

A mid-plane cross-section of the generated structure. Black = solid, white = pore. The random morphology is qualitatively similar to what you would see in a segmented tomography image.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(micro[N // 2, :, :], cmap="gray", interpolation="nearest")
ax.set_title(f"Mid-plane slice (z = {N // 2})")
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Computing Transport Properties

We now pass the NumPy array directly into OpenImpala. The Python bindings transfer the array into an AMReX `iMultiFab`, formulate the steady-state diffusion problem, and solve it with HYPRE.

We set `max_grid_size=128` because our $100^3$ domain fits comfortably in memory on a single node. This avoids unnecessary domain decomposition overhead for small problems. For guidance on tuning this parameter at larger scales, see [Tutorial 7](07_hpc_scaling.ipynb).

In [ ]:
print("Starting physics calculations...")

# 1. Volume fraction
vf = oi.volume_fraction(micro, phase=1)
print(f"\nPorosity (phase 1 VF): {vf.fraction:.4f}")

# 2. Percolation check (does the pore space span the domain?)
t0 = time.time()
perc = oi.percolation_check(micro, phase=1, direction="z")
print(f"Percolates in Z:       {perc.percolates} ({time.time()-t0:.3f}s)")

# 3. Solve for tortuosity
if perc.percolates:
    t0 = time.time()
    result = oi.tortuosity(micro, phase=1, direction="z", solver="flexgmres", max_grid_size=128)
    t_solve = time.time() - t0

    print(f"\nTortuosity (tau):      {result.tortuosity:.4f}")
    print(f"Solver iterations:     {result.iterations} (Residual: {result.residual_norm:.2e})")
    print(f"Wall time:             {t_solve:.2f}s for {micro.size / 1e6:.1f}M voxels")
else:
    print("Phase does not percolate -- tortuosity is undefined.")

## 3. Parametric Sweep: Tortuosity vs. Porosity

Because OpenImpala operates directly on NumPy arrays, scripting parametric studies is straightforward. Below we sweep over a range of porosities. At low porosity the pore network may become disconnected and the phase will not percolate — in that case, tortuosity is undefined and we skip that point.

In [ ]:
porosities = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
results = []  # (porosity, tau) pairs

print("Running parametric sweep...")
sweep_start = time.time()

for target in porosities:
    # Use a smaller grid (64^3) for the sweep to keep run times short
    if HAS_PORESPY:
        data = ps.generators.blobs(
            shape=[64, 64, 64], porosity=target, blobiness=1.5,
        ).astype(np.int32)
    else:
        # Seed per-porosity for reproducibility; uncorrelated voxels can
        # produce pathologically thin paths at low porosity, so we also
        # apply a mild dilation to improve connectivity.
        rng = np.random.default_rng(seed=int(target * 100))
        raw = rng.random(size=(64, 64, 64))
        data = (raw < target).astype(np.int32)

    perc = oi.percolation_check(data, phase=1, direction="z")
    if not perc.percolates:
        print(f"  Porosity={target:.2f}  ->  Does not percolate (skipped)")
        continue
    try:
        res = oi.tortuosity(data, phase=1, direction="z", solver="flexgmres", max_grid_size=128)
        results.append((target, res.tortuosity))
        print(f"  Porosity={target:.2f}  ->  Tau={res.tortuosity:.4f}")
    except oi.ConvergenceError:
        print(f"  Porosity={target:.2f}  ->  Solver did not converge (skipped)")

print(f"\nEvaluated {len(porosities)} microstructures in {time.time() - sweep_start:.2f}s.")

# Plot the sweep results
if results:
    phi_vals, tau_vals = zip(*results)

    fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
    ax.plot(phi_vals, tau_vals, "o-", color="#d62728", linewidth=2.5, markersize=8,
            markerfacecolor="white", markeredgewidth=2)

    ax.set_xlabel(r"Porosity ($\varepsilon$)", fontsize=12)
    ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontsize=12)
    ax.set_title("Random Porous Media: Tortuosity vs. Porosity")
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

## 4. Loading Your Own Data

The examples above used synthetic microstructures. When you have a real segmented 3D image (from X-ray CT, FIB-SEM, etc.), the workflow is the same — load it into a NumPy array, then pass it to OpenImpala.

```python
# TIFF stack (most common format from tomography)
import tifffile
data = tifffile.imread("my_scan.tif").astype(np.int32)
result = oi.tortuosity(data, phase=1, direction="z")

# HDF5 dataset
import h5py
with h5py.File("my_scan.hdf5", "r") as f:
    data = f["/exchange/data"][:].astype(np.int32)

# RAW binary (flat file, you must know the dimensions)
data = np.fromfile("my_scan.raw", dtype=np.uint8).reshape((Nz, Ny, Nx))
data = (data > 128).astype(np.int32)  # threshold to binary
```

**Key points:**
- The array must be `int32` with integer phase labels (0, 1, 2, ...).
- If your image is greyscale, **threshold it first** to produce a segmented phase map.
- The array shape convention is **(Z, Y, X)** — the first axis is the stack direction.
- For very large files that don't fit in RAM, use OpenImpala's built-in parallel C++ readers via `oi.read_image()` — see [Tutorial 7](07_hpc_scaling.ipynb).

[Tutorial 2](02_digital_twin.ipynb) demonstrates a full worked example with a real TIFF stack.

## Next Steps

In this tutorial we generated a microstructure, solved for its transport properties, and ran a parametric sweep — all from a single Python session, with no mesh generation or input files needed.

**Up next in the series:**
- [Tutorial 2: From 3D Image to Device Model](02_digital_twin.ipynb) — Load a real CT scan, measure anisotropy, and export parameters to PyBaMM.
- [Tutorial 3: REV and Uncertainty](03_rev_and_uncertainty.ipynb) — Determine whether your sample volume is statistically representative.
- [Tutorial 4: Effective Diffusivity and Field Visualisation](04_multiphase_and_fields.ipynb) — Use the cell-problem solver and visualise internal fields.
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Generate training data for machine learning models.
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) — Use OpenImpala inside an optimisation loop.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Deploy on clusters with MPI.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **AMReX:** W. Zhang et al., *AMReX: a framework for block-structured adaptive mesh refinement*, J. Open Source Software 4(37), 1370 (2019). [doi:10.21105/joss.01370](https://doi.org/10.21105/joss.01370)
3. **HYPRE:** R. D. Falgout & U. M. Yang, *hypre: A library of high performance preconditioners*, Computational Science — ICCS 2002, LNCS 2331, pp. 632–641 (2002). [doi:10.1007/3-540-47789-6_66](https://doi.org/10.1007/3-540-47789-6_66)
4. **PoreSpy:** J. T. Gostick et al., *PoreSpy: A Python toolkit for quantitative analysis of porous media images*, J. Open Source Software 4(37), 1296 (2019). [doi:10.21105/joss.01296](https://doi.org/10.21105/joss.01296)
5. **Tortuosity in porous media:** B. Tjaden et al., *On the origin and application of the Bruggeman correlation for analysing transport phenomena in electrochemical systems*, Current Opinion in Chemical Engineering 12, 44–51 (2016). [doi:10.1016/j.coche.2016.02.006](https://doi.org/10.1016/j.coche.2016.02.006)